[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jgcalvo/SP1306/blob/main/edps-hiperbolicas/demo_onda2d.ipynb)

# Ecuación de onda en 2D con elementos P1

Traducción de `demo_onda2d.m`.

$$u_{tt}=\Delta u \quad\text{en } (0,1)^2\times(0,T],\qquad u=0 \text{ en }\partial U,
\qquad u(\cdot,0)=g,\quad u_t(\cdot,0)=0 .$$

Con $c=1$ y $g(x)=\operatorname{sen}(3\pi x_1)\operatorname{sen}(4\pi x_2)$ la solución exacta es

$$u(x,t)=\operatorname{sen}(3\pi x_1)\operatorname{sen}(4\pi x_2)\cos(5\pi t),$$

porque $\Delta u=-(9+16)\pi^2u=-25\pi^2u=u_{tt}$. El periodo es $2/5$.

El cuaderno hace tres cosas:

1. arma las matrices de masa y rigidez $P_1$, y calcula la condición CFL del esquema explícito;
2. corre las tres discretizaciones en tiempo y las compara con la solución exacta;
3. barre $\Delta t$ para ver exactamente dónde explota cada una.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from scipy.sparse import coo_matrix
from scipy.sparse.linalg import splu, eigsh
from IPython.display import HTML

## La malla y el ensamblaje

Triangulación uniforme del cuadrado: cada celda se parte en dos triángulos. El ensamblaje
es el mismo de siempre, con

$$M_{ij}=\int_U\varphi_i\varphi_j,\qquad A_{ij}=\int_U\nabla\varphi_i\cdot\nabla\varphi_j,$$

y al final se eliminan las filas y columnas de los nodos de frontera, donde $u=0$.
Como en NumPy los índices arrancan en 0, los números de `elem` aparecen corridos en 1
respecto al `.m`.

In [ ]:
def mallaP1(h):
    """Matrices de masa y rigidez P1 en (0,1)^2, restringidas a nodos interiores."""
    x = np.arange(0, 1 + h/2, h); n = x.size
    X, Y = np.meshgrid(x, x, indexing='ij')
    node = np.column_stack([X.ravel(order='F'), Y.ravel(order='F')])
    idx  = lambda i, j: j*n + i
    elem = []
    for j in range(n-1):
        for i in range(n-1):
            elem.append([idx(i, j), idx(i+1, j),   idx(i+1, j+1)])
            elem.append([idx(i, j), idx(i+1, j+1), idx(i, j+1)  ])
    elem = np.array(elem); NT, N = elem.shape[0], node.shape[0]

    ve = np.zeros((NT, 2, 3))                 # ve[:,:,i]: lado opuesto al vertice i
    ve[:, :, 2] = node[elem[:, 1]] - node[elem[:, 0]]
    ve[:, :, 0] = node[elem[:, 2]] - node[elem[:, 1]]
    ve[:, :, 1] = node[elem[:, 0]] - node[elem[:, 2]]
    area = .5*(-ve[:, 0, 2]*ve[:, 1, 1] + ve[:, 1, 2]*ve[:, 0, 1])

    ii, jj, sA, sM = [], [], [], []
    for a in range(3):
        for b in range(3):
            ii.append(elem[:, a]); jj.append(elem[:, b])
            sA.append(np.einsum('ij,ij->i', ve[:, :, a], ve[:, :, b])/(4*area))
            sM.append(area*((a == b) + 1)/12)
    ii, jj = np.concatenate(ii), np.concatenate(jj)
    A = coo_matrix((np.concatenate(sA), (ii, jj)), shape=(N, N)).tocsc()
    M = coo_matrix((np.concatenate(sM), (ii, jj)), shape=(N, N)).tocsc()

    bd  = (node[:, 0] == 0) | (node[:, 0] == 1) | (node[:, 1] == 0) | (node[:, 1] == 1)
    inn = ~bd
    return M[inn][:, inn], A[inn][:, inn], node[inn], node, elem, inn

## Datos y condición CFL

El esquema explícito de diferencias centradas es estable si y solo si

$$\Delta t\ \leq\ \Delta t_{\text{CFL}}=\frac{2}{\sqrt{\lambda_{\max}}},\qquad
A\bm w=\lambda M\bm w .$$

Para elementos $P_1$ se tiene $\lambda_{\max}=\mathcal{O}(h^{-2})$, así que esto es
$\Delta t=\mathcal{O}(h)$. La constante $\lambda_{\max}h^2$ que se imprime abajo es la de la
desigualdad inversa, y depende solo del elemento y de la malla.

In [ ]:
h = 1/32
M, A, xy, node, elem, inn = mallaP1(h)

uex = lambda t: np.sin(3*np.pi*xy[:, 0])*np.sin(4*np.pi*xy[:, 1])*np.cos(5*np.pi*t)
U0  = uex(0.0)
V0  = np.zeros_like(U0)

lmax  = eigsh(A, k=1, M=M, which='LM', return_eigenvectors=False)[0]
dtCFL = 2/np.sqrt(lmax)

print(f"h = 1/{round(1/h)},  {U0.size} nodos interiores")
print(f"lambda_max = {lmax:.4e}      lambda_max*h^2 = {lmax*h**2:.2f}")
print(f"dt_CFL = 2/sqrt(lambda_max) = {dtCFL:.4e}")

## Los tres integradores

Los tres son puntos de la familia de **Newmark-$\beta$**:

| $(\beta,\gamma)$ | método | tipo | orden |
|---|---|---|---|
| $(0,\tfrac12)$ | diferencias centradas | explícito, condicional | 2 |
| $(\tfrac14,\tfrac12)$ | Crank–Nicolson | implícito, incondicional | 2 |
| $(0{,}3025\,;\,0{,}6)$ | Newmark disipativo | implícito, incondicional | 1 |

Cada paso resuelve un sistema con $K_\beta=M+\beta\Delta t^2A$, que no depende de $n$: se
factoriza una sola vez. Con $\beta=0$ esa matriz es $M$ sola.

In [ ]:
def newmark(M, A, U0, V0, dt, nt, be, ga):
    """Familia de Newmark-beta para M U'' + A U = 0, guardando la historia."""
    lu_M = splu(M)                                   # para las aceleraciones
    U, V = U0.copy(), V0.copy()
    Ac   = lu_M.solve(-(A @ U))
    lu_K = lu_M if be == 0 else splu((M + be*dt**2*A).tocsc())

    Uh = np.zeros((U0.size, nt+1)); Vh = np.zeros_like(Uh)
    Uh[:, 0], Vh[:, 0] = U0, V0
    tv = np.arange(nt+1)*dt

    for k in range(1, nt+1):
        Ut = U + dt*V + (dt**2/2)*(1 - 2*be)*Ac      # predictor
        Un = lu_K.solve(M @ Ut)
        An = (Un - Ut)/(be*dt**2) if be > 0 else lu_M.solve(-(A @ Un))
        V  = V + dt*((1 - ga)*Ac + ga*An)            # corrector
        U, Ac = Un, An
        if not np.all(np.isfinite(U)):
            Uh[:, k:] = np.inf; Vh[:, k:] = np.inf
            return Uh, Vh, tv
        Uh[:, k], Vh[:, k] = U, V
    return Uh, Vh, tv

## Las tres discretizaciones contra la solución exacta

Se compara cada método con el mayor paso que puede permitirse. Las dos primeras filas usan
el **mismo** $\Delta t$ y dan prácticamente lo mismo; las dos últimas muestran qué pasa al
tomar $\Delta t$ diez veces mayor que la CFL, que es lo único que los implícitos compran.

In [ ]:
T = 0.8                                   # dos periodos
casos = [("Centradas,  dt = 0.9 CFL",        0.0,   .5, 0.9*dtCFL),
         ("Crank-Nicolson, dt = 0.9 CFL",    .25,   .5, 0.9*dtCFL),
         ("Crank-Nicolson, dt = 10 CFL",     .25,   .5,  10*dtCFL),
         ("Newmark (.3025,.6), dt = 10 CFL", .3025, .6,  10*dtCFL)]

print(f"{'metodo':34s}{'dt':>11s}{'pasos':>7s}{'err final':>13s}{'E(T)/E(0)':>12s}")
res = []
for nom, be, ga, dt0 in casos:
    nt = round(T/dt0); dt = T/nt
    Uh, Vh, tv = newmark(M, A, U0, V0, dt, nt, be, ga)
    err = np.array([np.max(np.abs(Uh[:, k] - uex(tv[k]))) for k in range(nt+1)])
    E   = np.array([.5*Vh[:, k] @ (M @ Vh[:, k]) + .5*Uh[:, k] @ (A @ Uh[:, k])
                    for k in range(nt+1)])
    res.append((nom, tv, err, E/E[0], Uh))
    print(f"{nom:34s}{dt:11.3e}{nt:7d}{err[-1]:13.3e}{E[-1]/E[0]:12.4f}")

In [ ]:
cols = ['tab:blue', 'tab:green', 'tab:orange', 'tab:red']
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for (nom, tv, err, E, _), c in zip(res, cols):
    ax[0].plot(tv, err, color=c, lw=1.7, label=nom)
    ax[1].plot(tv, E,   color=c, lw=1.7, label=nom)
ax[0].set_xlabel('t'); ax[0].set_ylabel(r'$\|u_h(t)-u(t)\|_\infty$')
ax[0].set_title('Error contra la solución exacta'); ax[0].grid(alpha=.3)
ax[0].legend(fontsize=7)
ax[1].axhline(1, ls=':', color='gray'); ax[1].set_ylim(0, 1.15)
ax[1].set_xlabel('t'); ax[1].set_ylabel(r'$E^n/E^0$')
ax[1].set_title('Energía discreta'); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## Estabilidad: ¿dónde explota?

Se corre cada método hasta el mismo $T$ con $\Delta t$ creciente, en múltiplos de
$\Delta t_{\text{CFL}}$, y se mira $\|\bm U^N\|_\infty$.

In [ ]:
pares = [(0.0, .5), (.25, .5), (.3025, .6)]
print(f"{'dt/CFL':>8s}{'dt':>12s}{'centradas':>14s}{'C-N':>14s}{'Newmark':>14s}")
for f in [0.5, 0.9, 0.99, 1.0, 1.01, 1.05, 1.2, 5, 50]:
    nt = max(2, round(T/(f*dtCFL))); dt = T/nt
    m  = [np.max(np.abs(newmark(M, A, U0, V0, dt, nt, b, g)[0][:, -1])) for b, g in pares]
    print(f"{f:8.2f}{dt:12.3e}{m[0]:14.3e}{m[1]:14.3e}{m[2]:14.3e}")

## El umbral, y por qué depende del horizonte

Afinando por bisección se obtiene el $\Delta t$ a partir del cual el explícito se desborda.
El umbral **exacto** es $\Delta t_{\text{CFL}}$; si se mira poco tiempo sale un poco mayor,
porque justo encima de la CFL el modo inestable tiene $|\rho|$ apenas mayor que $1$ y no le
da tiempo de crecer.

In [ ]:
print("umbral empirico del explicito (criterio ||U||_inf > 10):")
for Tu in (T, 5*T):
    lo, hi = 0.90, 1.10
    while hi - lo > 1e-4:
        f  = (lo + hi)/2
        nt = round(Tu/(f*dtCFL))
        U  = newmark(M, A, U0, V0, Tu/nt, nt, 0, .5)[0][:, -1]
        if np.max(np.abs(U)) > 10: hi = f
        else:                      lo = f
    print(f"   mirando hasta T = {Tu:4.1f} :  dt = {lo:.4f} dt_CFL")

### Lecturas

- El explícito pasa de $\mathcal{O}(1)$ a desbordarse **justo** al cruzar la CFL.
- Los implícitos nunca se desbordan, pero eso no los vuelve correctos: con
  $\Delta t=50\,\Delta t_{\text{CFL}}$ la amplitud es $0{,}35$ y $0{,}03$ en vez de $\approx1$.
  Crank–Nicolson se desfasa; Newmark además amortigua.
- Cuidado con las dos últimas filas del cuadro: con $\Delta t$ tan grande el explícito da solo
  dos pasos y no alcanza a desbordarse, pero el número que imprime tampoco significa nada.

## La onda en movimiento

Animación del esquema explícito a $0{,}9\,\Delta t_{\text{CFL}}$ (el primer caso del cuadro).

In [ ]:
_, tv, _, _, Uh = res[0]
frames = range(0, len(tv), 4)

u_full = np.zeros(node.shape[0])
fig = plt.figure(figsize=(6, 4.5))
ax  = fig.add_subplot(projection='3d')

def actualizar(j):
    ax.clear()
    u_full[inn] = Uh[:, j]
    ax.plot_trisurf(node[:, 0], node[:, 1], u_full, triangles=elem,
                    cmap='viridis', linewidth=0, antialiased=False)
    ax.set_zlim(-1.05, 1.05); ax.set_xlabel('x_1'); ax.set_ylabel('x_2')
    ax.set_title(f't = {tv[j]:.3f}')

ani = animation.FuncAnimation(fig, actualizar, frames=frames, interval=80, blit=False)
plt.close(fig)
HTML(ani.to_jshtml())